In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Die Onkologie-Station

Auf einer Onkologie-Station liegen fünf Patienten in nebeneinander liegenden Zimmern.
Bis auf einen  der Patienten hat jeder genau eine Zigaretten-Marke geraucht.
Der Patient, der nicht Zigarette geraucht hat, hat Pfeife geraucht.
Jeder Patient fährt genau ein Auto und ist
an genau einer Krebs-Art erkrankt.  Zusätzlich haben Sie die folgenden Informationen:
<ol>
<li> Im Zimmer neben Michael wird Camel geraucht. </li>
<li> Der Trabant-Fahrer raucht Ernte 23 und liegt im Zimmer neben dem 
      Zungen-Krebs Patienten. </li>
<li> Rolf liegt im letzten Zimmer und hat Kehlkopf-Krebs. </li>
<li> Der West-Raucher liegt im ersten Zimmer. </li>
<li> Der Mazda-Fahrer hat Zungen-Krebs und liegt neben dem Trabant-Fahrer. </li>
<li> Der Nissan-Fahrer liegt neben dem Zungen-Krebs Patient. </li>
<li> Rudolf wünscht sich Sterbe-Hilfe und liegt zwischen dem Camel-Raucher und dem Trabant-Fahrer. </li>
<li> Der Seat Fahrer hat morgen seinen letzten Geburtstag. </li>
<li> Der Luckies Raucher liegt neben dem Patienten mit Lungen-Krebs. </li>
<li> Der Camel Raucher liegt neben dem Patienten mit Darm-Krebs. </li>
<li> Der Nissan Fahrer liegt neben dem Mazda-Fahrer. </li>
<li> Der Mercedes-Fahrer raucht Pfeife und liegt neben dem Camel Raucher. </li>
<li> Jens liegt neben dem Luckies Raucher. </li>
<li> Der Hodenkrebs-Patient hat gestern seine Eier durchs Klo gespült. </li>
</ol>
Entwickeln Sie ein <em>F#</em>-Programm, das die folgenden Fragen beantwortet:
<ol> 
<li> Was raucht der Darmkrebs-Patient? </li>
<li> Was fährt Kurt für ein Auto? </li>
</ol>

## Importing the Necessary Notebooks

We import the backtracking solver.  The notebook `02-Backtracking-Constraint-Solver.ipynb` defines the types `Constraint<'V>` and `CSP<'V>`, the functions `constr`, `different`, `equal`, and `hasValue` that create constraints, and the function `solve`.  We use one variable for every property, e.g. the variable `Camel` is the number of the room of the patient who smokes Camel.

In [2]:
importNotebook "02-Backtracking-Constraint-Solver.ipynb"

imported 

02-Backtracking-Constraint-Solver.ipynb

## Auxiliary Functions

In order to succinctly express the constraints that every patient has a different kind of cancer, drives a different brand of car etc., we use the function `allDifferent`.  It takes one argument:
- `Variables` is a list of variables.

It returns a list of constraints that is satisfied if and only if all the variables from `Variables` have different values.

In [3]:
let allDifferent (Variables: string list) : Constraint<'V> list =
    [ for x in Variables do
        for y in Variables do
            if x < y then different x y ]

In [4]:
(allDifferent [ "a"; "b"; "c" ] : Constraint<int> list)

[a ≠ b; a ≠ c; b ≠ c]

The function `nextTo` takes two arguments:
- `x` is a variable,
- `y` is a variable.

It returns a constraint that is satisfied if the rooms `x` and `y` are neighbours, i.e. if $|x - y| = 1$.

In [5]:
let nextTo (x: string) (y: string) : Constraint<int> =
    constr (sprintf "|%s - %s| = 1" x y) [x; y] (fun A -> abs (A[x] - A[y]) = 1)

In [6]:
let Brands  = [ "Camel"; "Ernte"; "West"; "Luckies"; "Pfeife" ]
let Cars    = [ "Trabant"; "Mazda"; "Nissan"; "Seat"; "Mercedes" ]
let Cancers = [ "Zunge"; "Kehlkopf"; "Lunge"; "Darm"; "Hoden" ]
let Names   = [ "Michael"; "Rolf"; "Rudolf"; "Jens"; "Kurt" ]

The function `createCSP` takes no arguments.  It returns a CSP that encodes the puzzle.  The order of the variables is chosen so that variables that are connected by constraints are assigned in succession.

In [7]:
let createCSP () : CSP<int> =
    let Variables = [ "Rolf"; "West"; "Kehlkopf"; "Michael"; "Camel"; "Trabant"; "Rudolf"; "Ernte"; "Zunge"; "Luckies"; "Pfeife"
                      "Mazda"; "Nissan"; "Seat"; "Mercedes"; "Lunge"; "Darm"; "Hoden"; "Jens"; "Kurt" ]
    let Values = [ 1 .. 5 ]
    // In jedem Zimmer wird genau eine Zigaretten-Marke geraucht.
    let mutable Constraints = allDifferent Brands
    // Jeder Patient hat genau ein Auto.
    Constraints <- Constraints @ allDifferent Cars
    // Jeder Patient ist an genau einer Krebsart erkrankt.
    Constraints <- Constraints @ allDifferent Cancers
    // Jeder Patient hat genau einen Vornamen.
    Constraints <- Constraints @ allDifferent Names
    // Im Zimmer neben Michael wird Camel geraucht. 
    Constraints <- Constraints @ [ nextTo "Michael" "Camel" ]
    // Der Trabant-Fahrer raucht Ernte 23 und liegt im Zimmer neben dem 
    // Zungen-Krebs Patienten. 
    Constraints <- Constraints @ [ equal "Trabant" "Ernte"; nextTo "Trabant" "Zunge" ]
    // Rolf liegt im letzten Zimmer und hat Kehlkopf-Krebs. 
    Constraints <- Constraints @ [ hasValue "Rolf" 5; equal "Rolf" "Kehlkopf" ]
    // Der West-Raucher liegt im ersten Zimmer. 
    Constraints <- Constraints @ [ hasValue "West" 1 ]
    // Der Mazda-Fahrer hat Zungen-Krebs und liegt neben dem Trabant-Fahrer. 
    Constraints <- Constraints @ [ equal "Mazda" "Zunge"; nextTo "Mazda" "Trabant" ]
    // Der Nissan-Fahrer liegt neben dem Zungen-Krebs Patient. 
    Constraints <- Constraints @ [ nextTo "Nissan" "Zunge" ]
    // Rudolf wünscht sich Sterbe-Hilfe und liegt zwischen dem Camel-Raucher und dem Trabant-Fahrer. 
    Constraints <- Constraints @ [ nextTo "Rudolf" "Camel"; nextTo "Rudolf" "Trabant"; different "Camel" "Trabant" ]
    // Der Luckies Raucher liegt neben dem Patienten mit Lungen-Krebs. 
    Constraints <- Constraints @ [ nextTo "Luckies" "Lunge" ]
    // Der Camel Raucher liegt neben dem Patienten mit Darm-Krebs. 
    Constraints <- Constraints @ [ nextTo "Camel" "Darm" ]
    // Der Nissan Fahrer liegt neben dem Mazda-Fahrer. 
    Constraints <- Constraints @ [ nextTo "Nissan" "Mazda" ]
    // Der Mercedes-Fahrer raucht Pfeife und liegt neben dem Camel Raucher. 
    Constraints <- Constraints @ [ equal "Mercedes" "Pfeife"; nextTo "Mercedes" "Camel" ]
    // Jens liegt neben dem Luckies Raucher. 
    Constraints <- Constraints @ [ nextTo "Jens" "Luckies" ]
    // Der Hodenkrebs-Patient hat gestern seine Eier durchs Klo gespült.
    { Variables = Variables; Values = Values; Constraints = Constraints }

In [8]:
let csp = createCSP ()
csp

{ Variables = ["Rolf"; "West"; "Kehlkopf"; "Michael"; "Camel"; "Trabant"; "Rudolf"; "Ernte"; "Zunge"; "Luckies"; "Pfeife"; "Mazda"; "Nissan"; "Seat"; "Mercedes"; "Lunge"; "Darm"; "Hoden"; "Jens"; "Kurt"]; Values = [1; 2; 3; 4; 5]; Constraints = [Camel ≠ Ernte; Camel ≠ West; Camel ≠ Luckies; Camel ≠ Pfeife; Ernte ≠ West; Ernte ≠ Luckies; Ernte ≠ Pfeife; Luckies ≠ West; Luckies ≠ Pfeife; Pfeife ≠ West; Mazda ≠ Trabant; Mazda ≠ Nissan; Mazda ≠ Seat; Mazda ≠ Mercedes; Nissan ≠ Trabant; Nissan ≠ Seat; Seat ≠ Trabant; Mercedes ≠ Trabant; Mercedes ≠ Nissan; Mercedes ≠ Seat; Kehlkopf ≠ Zunge; Kehlkopf ≠ Lunge; Lunge ≠ Zunge; Darm ≠ Zunge; Darm ≠ Kehlkopf; Darm ≠ Lunge; Darm ≠ Hoden; Hoden ≠ Zunge; Hoden ≠ Kehlkopf; Hoden ≠ Lunge; Michael ≠ Rolf; Michael ≠ Rudolf; Rolf ≠ Rudolf; Jens ≠ Michael; Jens ≠ Rolf; Jens ≠ Rudolf; Jens ≠ Kurt; Kurt ≠ Michael; Kurt ≠ Rolf; Kurt ≠ Rudolf; |Michael - Camel| = 1; Trabant = Ernte; |Trabant - Zunge| = 1; Rolf = 5; Rolf = Kehlkopf; West = 1; Mazda = Zunge; |Ma

With the right ordering of the variables, solving the problem takes only a few milliseconds.

In [9]:
#!time
let Solution = (solve csp).Value

Wall time: 5.2088ms

In [10]:
Solution

map [("Camel", 3); ("Darm", 2); ("Ernte", 5); ("Hoden", 1); ("Jens", 3); ("Kehlkopf", 5); ("Kurt", 1); ("Luckies", 4); ("Lunge", 3); ("Mazda", 4); ("Mercedes", 2); ("Michael", 2); ("Nissan", 3); ("Pfeife", 2); ("Rolf", 5); ("Rudolf", 4); ("Seat", 1); ("Trabant", 5); ("West", 1); ("Zunge", 4)]

## Pretty Printing the Solution

The function `showHTML` takes one argument:
- `Solution` is a map assigning room numbers to all properties.

It returns an HTML table that shows the solution.  The table has one row for every room.

In [11]:
let showHTML (Solution: Map<string, int>) =
    let sb = System.Text.StringBuilder()
    sb.Append("<table style=\"border:2px solid blue\">\n<tr>") |> ignore
    for name in [ "Room"; "Brand"; "Car"; "Cancer"; "Name" ] do
        sb.Append("<th style=\"color:gold; background-color:blue\">" + name + "</th>") |> ignore
    sb.Append("</tr>\n") |> ignore
    for room in 1 .. 5 do
        sb.Append("<tr><td style=\"border:1px solid green\">" + string room + "</td>") |> ignore
        for Class in [ Brands; Cars; Cancers; Names ] do
            for x in Class do
                if Solution[x] = room then
                    sb.Append("<td style=\"border:1px solid green\">" + x + "</td>") |> ignore
        sb.Append("</tr>\n") |> ignore
    sb.Append("</table>") |> ignore
    HTML(sb.ToString())

In [12]:
showHTML Solution

Room,Brand,Car,Cancer,Name
1,West,Seat,Hoden,Kurt
2,Pfeife,Mercedes,Darm,Michael
3,Camel,Nissan,Lunge,Jens
4,Luckies,Mazda,Zunge,Rudolf
5,Ernte,Trabant,Kehlkopf,Rolf


## Checking the Uniqueness

The function `negateSolution` takes one argument:
- `Solution` is a map that represents a solution.

It returns a constraint that is satisfied iff one of the variables has a value that is different from its value in `Solution`.

In [13]:
let negateSolution (Solution: Map<string, int>) : Constraint<int> =
    let vars = [ for KeyValue(x, _) in Solution -> x ]
    let text = String.concat " ∨ " [ for KeyValue(x, v) in Solution -> sprintf "%s ≠ %d" x v ]
    constr text vars (fun A -> vars |> List.exists (fun x -> A[x] <> Solution[x]))

In [14]:
negateSolution Solution

Camel ≠ 3 ∨ Darm ≠ 2 ∨ Ernte ≠ 5 ∨ Hoden ≠ 1 ∨ Jens ≠ 3 ∨ Kehlkopf ≠ 5 ∨ Kurt ≠ 1 ∨ Luckies ≠ 4 ∨ Lunge ≠ 3 ∨ Mazda ≠ 4 ∨ Mercedes ≠ 2 ∨ Michael ≠ 2 ∨ Nissan ≠ 3 ∨ Pfeife ≠ 2 ∨ Rolf ≠ 5 ∨ Rudolf ≠ 4 ∨ Seat ≠ 1 ∨ Trabant ≠ 5 ∨ West ≠ 1 ∨ Zunge ≠ 4

The function `checkUniqueness` takes two arguments:
* `Solution` is a solution of `csp`.
* `csp` is a *constraint satisfaction problem*.

It tries to compute a new solution for the constraint satisfaction problem that is different from the given solution.  If there is such a solution, it is displayed.

In [15]:
let checkUniqueness (Solution: Map<string, int>) (csp: CSP<int>) =
    let NewCSP = { csp with Constraints = csp.Constraints @ [ negateSolution Solution ] }
    match solve NewCSP with
    | Some other ->
        printfn "The solution is not unique. The alternative solution is:"
        display (showHTML other) |> ignore
    | None ->
        printfn "Well done! The solution is unique."

In [16]:
checkUniqueness Solution csp

Well done! The solution is unique.